# BITS Academic Course Recommender — FINAL Colab Notebook

**Single-notebook build. No Streamlit. No project-module imports. No ngrok.**

This notebook is designed to run directly in Google Colab and uses the supplied BITS handout PDFs as the source for course-preference facts. It can also ingest the current Bulletin, Academic Regulations and timetable when those PDFs are uploaded.

### What this version does

`Handout ZIP → structured course data → deterministic academic gate → natural-language preference matching → source-backed recommendation cards`

The recommendation layer **does not invent** attendance, exam, makeup or academic-rule facts. `unknown` stays `unknown`.

### Files accepted

- One handout ZIP containing course PDFs (your current archive has 540 PDFs).
- Optional `bulletin.pdf`.
- Optional `Academic-Regulations-2023.pdf`.
- Optional `timetable.pdf`.

The notebook still runs when only the handout ZIP is available. In that case, it will explicitly say which academic-rule facts could not be verified rather than pretending they are known.

In [ ]:
# CELL 1 — Install only the small set of dependencies this notebook needs.
# No Streamlit, no ngrok, no separate project package.

import sys, subprocess
req = ["PyMuPDF>=1.24,<2", "ipywidgets>=8,<9"]
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *req], check=True)
else:
    print("Non-Colab test mode: using packages already installed; Colab will install PyMuPDF + ipywidgets automatically.")

In [ ]:
# CELL 2 — Runtime paths + Colab/local input discovery.
from pathlib import Path
import os, re, json, zipfile, shutil, hashlib, math, unicodedata, html
from collections import Counter, defaultdict

IS_COLAB = "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", "")) or Path("/content").exists()
ROOT = Path("/content/bits_r2_final") if Path("/content").exists() else Path("/mnt/data/bits_r2_final")
RAW = ROOT / "data" / "raw"
HANDOUTS = RAW / "handouts"
VERIFIED = ROOT / "data" / "verified"
OUT = ROOT / "data" / "processed"
for p in [RAW, HANDOUTS, VERIFIED, OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("COLAB detected:", IS_COLAB)

In [ ]:
# CELL 3 — Upload input files.
# In Colab this opens the normal upload dialog only when the needed files are absent.
# The current handout archive is accepted as-is; do not rename or re-compress it first.

from pathlib import Path

handout_zips = list(Path("/content").glob("*.zip")) if Path("/content").exists() else []
source_pdfs = {
    "bulletin": Path("/content/bulletin.pdf"),
    "regulations": Path("/content/Academic-Regulations-2023.pdf"),
    "timetable": Path("/content/timetable.pdf"),
}

if IS_COLAB:
    need_upload = not handout_zips
    if need_upload:
        from google.colab import files
        print("Upload your handout ZIP now (the 540-PDF archive is fine).")
        uploaded = files.upload()
        handout_zips = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
    else:
        print("Handout ZIP already present:", handout_zips[0].name)

    missing_sources = [k for k,p in source_pdfs.items() if not p.exists()]
    if missing_sources:
        print("\nOptional source PDFs missing:", ", ".join(missing_sources))
        print("You may upload them now. If you skip them, the handout recommender still works.")
        # Uploading is optional. The next cell copies whatever is present.
        uploaded2 = files.upload() if len(missing_sources) else {}
else:
    handout_zips = [Path("/mnt/data/handouts(1).zip")] if Path("/mnt/data/handouts(1).zip").exists() else []
    print("Local test mode. Handout ZIP candidates:", [p.name for p in handout_zips])

if not handout_zips:
    raise FileNotFoundError("No handout ZIP found. Upload the handout archive and rerun this cell.")

HANDOUT_ZIP = handout_zips[0]
print("\nUsing handout ZIP:", HANDOUT_ZIP)
print("Size (MB):", round(HANDOUT_ZIP.stat().st_size/1024/1024, 1))

In [ ]:
# CELL 4 — Extract the handout archive safely.
# Re-running this cell wipes only this notebook's extracted handout folder.

if HANDOUTS.exists():
    shutil.rmtree(HANDOUTS)
HANDOUTS.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(HANDOUT_ZIP) as z:
    pdf_infos = [info for info in z.infolist() if info.filename.lower().endswith(".pdf") and not info.is_dir()]
    for info in pdf_infos:
        target = HANDOUTS / Path(info.filename).name
        with z.open(info) as src, target.open("wb") as dst:
            shutil.copyfileobj(src, dst)

pdf_paths = sorted(HANDOUTS.glob("*.pdf"))
print("PDFs extracted:", len(pdf_paths))
if not pdf_paths:
    raise RuntimeError("The ZIP contains no PDFs.")

## Parsing rules

The parser is intentionally conservative:

- Course codes are normalized but cross-listed aliases are retained.
- Titles/instructors are taken from explicit labelled fields when present.
- Missing course-specific attendance stays `unknown`.
- `no midsem` or `no compre` is recognized only when the handout explicitly says so.
- A makeup policy is stored as source text; it is **not** subjectively labelled “lenient”.
- Extraction confidence is stored so weak records can be flagged instead of silently trusted.

In [ ]:
# CELL 5 — Robust handout parser.
import pymupdf

COURSE_CODE_RE = re.compile(r"\b[A-Z]{2,10}\s+[A-Z]\s?\d{3,4}[A-Z]?\b", re.I)
FILE_CODE_RE = re.compile(r"^\d+_([A-Z0-9]+)_([A-Z]\d{3,4}[A-Z]?)\.pdf$", re.I)


def sanitize_text(s: str) -> str:
    s = s or ""
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Cf")
    return re.sub(r"\s+", " ", s).strip()


def clean_space(s: str) -> str:
    return sanitize_text(s)


def norm_code(s: str) -> str:
    s = clean_space(s).upper()
    s = re.sub(r"\s+([A-Z])\s?(\d)", r" \1\2", s)
    return re.sub(r"\s+", " ", s)


def unique(seq):
    seen = set(); out = []
    for x in seq:
        if x and x not in seen:
            seen.add(x); out.append(x)
    return out


def course_codes_from_text(s: str):
    return unique([norm_code(m.group(0)) for m in COURSE_CODE_RE.finditer(s or "")])


def filename_code(path: Path):
    m = FILE_CODE_RE.match(path.name)
    return norm_code(f"{m.group(1)} {m.group(2)}") if m else None


def label_value(text: str, labels):
    # Handles both one-line and label-on-one-line / value-on-next-line layouts.
    for lab in labels:
        patterns = [
            rf"(?im)^\s*{lab}\s*[:\-]\s*([^\n]+)",
            rf"(?im)^\s*{lab}\s*\n\s*[:\-]?\s*([^\n]+)",
            rf"(?im){lab}\s*[:\-]\s*([^\n]+)",
        ]
        for pat in patterns:
            m = re.search(pat, text)
            if m:
                val = clean_space(m.group(1))
                if val:
                    return val
    return None


def find_page(pages, patterns):
    for i, t in enumerate(pages):
        if any(re.search(p, t, re.I) for p in patterns):
            return i + 1
    return None


def snippet_near(text, pattern, radius=1000):
    m = re.search(pattern, text, re.I | re.S)
    if not m:
        return None
    a = max(0, m.start() - radius // 4)
    b = min(len(text), m.end() + radius)
    return clean_space(text[a:b])


def tri_state(text, yes_patterns, no_patterns):
    for p in no_patterns:
        if re.search(p, text, re.I | re.S):
            return "no"
    for p in yes_patterns:
        if re.search(p, text, re.I | re.S):
            return "yes"
    return "unknown"


def extract_handout(pdf_path: Path):
    with pymupdf.open(pdf_path) as doc:
        pages = [page.get_text("text") for page in doc]
    text = "\n".join(pages)
    first = "\n".join(pages[:2])

    first_clean = "".join(ch for ch in first if unicodedata.category(ch) != "Cf")
    raw_code = label_value(first_clean, [r"Course\s*No\.?", r"Course\s*Number", r"Course\s*Code"])
    if not raw_code:
        m_ct = re.search(r"(?im)^\s*Course\s+Number\s*&\s*Title\s*[:\-]\s*(.+)$", first_clean)
        if m_ct:
            raw_code = m_ct.group(1)
    codes = course_codes_from_text(raw_code or "")
    file_code = filename_code(pdf_path)
    if file_code and file_code not in codes:
        codes.insert(0, file_code)
    primary = codes[0] if codes else file_code

    title = label_value(first_clean, [r"Course\s*Title", r"Name\s+of\s+the\s+course", r"Course\s*Name", r"Title\s+of\s+the\s+Course"])
    if not title:
        m_ct = re.search(r"(?im)^\s*Course\s+Number\s*&\s*Title\s*[:\-]\s*.+?\b[A-Z]{1,10}\s*[A-Z]\s?\d{3,4}[A-Z]?\s+(.+)$", first_clean)
        if m_ct:
            title = clean_space(m_ct.group(1))
    instructor = label_value(first_clean, [r"Instructor-in-Charge", r"Instructor-in-charge", r"Instructor\s+In\s+Charge", r"Instructor\s*\(s\)", r"Instructor\s+In\s+Charge"])

    # Fallback: some handouts place the first course summary in a compressed layout.
    if not title:
        m = re.search(r"(?is)Course\s+(?:Title|Name)\s*[:\-]?\s*([A-Za-z][^\n]{2,100})", first_clean)
        if m:
            title = clean_space(m.group(1))

    desc = snippet_near(text, r"(?:Course\s+Description|Catalog\s+Description|Introduction)[:\-]?", 1400)
    objectives = snippet_near(text, r"(?:Scope\s+and\s+Objective(?:s)?(?:\s+of\s+the\s+Course)?|Course\s+Objectives?)[:\-]?", 1200)
    prereq = snippet_near(text, r"PREREQUISITES?\*?\b", 1200)
    evaluation = snippet_near(text, r"(?:Evaluation\s+Scheme|Evaluation\s+Component|Evaluation\s+Pattern|Evaluation\s+Methodology)[:\-]?", 1800)
    makeup = snippet_near(text, r"make[- ]?up", 1100)

    has_midsem = tri_state(
        text,
        [r"\bmid[- ]?semester\b", r"\bmid[- ]?sem\b", r"\bmid\s*sem\b"],
        [r"\b(?:no|without)\s+(?:a\s+)?mid[- ]?(?:semester|sem)(?:\s+test|\s+exam(?:ination)?)?\b"]
    )
    has_compre = tri_state(
        text,
        [r"\bcomprehensive\s+(?:exam(?:ination)?|test)\b", r"\bcompre\b"],
        [r"\b(?:no|without)\s+(?:a\s+)?comprehensive\s+(?:exam(?:ination)?|test)\b"]
    )
    attendance = tri_state(
        text,
        [r"attendance[^\.\n]{0,140}(?:mandatory|compulsory|required|minimum|at\s+least\s+\d+\s*%)", r"(?:minimum|at\s+least)\s+\d+\s*%[^\.\n]{0,100}attendance"],
        [r"(?:attendance\s+is\s+not\s+(?:mandatory|required)|no\s+attendance\s+requirement|attendance\s+not\s+required)"]
    )

    eval_text = evaluation or text
    projects = "yes" if re.search(r"\bproject(?:s)?\b", eval_text, re.I) else "unknown"
    seminars = "yes" if re.search(r"\bseminar(?:s)?\b", eval_text, re.I) else "unknown"
    quizzes = "yes" if re.search(r"\bquiz(?:zes)?\b", eval_text, re.I) else "unknown"
    assignments = "yes" if re.search(r"\bassignment(?:s)?\b", eval_text, re.I) else "unknown"

    conf = 1.0
    if not primary: conf -= 0.45
    if not title: conf -= 0.35
    if len(sanitize_text(text)) < 700: conf -= 0.15
    if not instructor: conf -= 0.05
    conf = round(max(0.0, min(1.0, conf)), 3)

    refs = {
        "course": find_page(pages, [r"Course\s+(?:No\.?|Number|Code)"]),
        "description": find_page(pages, [r"Course\s+Description", r"Catalog\s+Description", r"Introduction"]),
        "evaluation": find_page(pages, [r"Evaluation\s+Scheme", r"Evaluation\s+Component", r"Evaluation\s+Pattern"]),
        "makeup": find_page(pages, [r"make[- ]?up"]),
        "prerequisites": find_page(pages, [r"PREREQUISITES?\*?"])
    }

    search_text = clean_space(" ".join(x for x in [title, primary, desc, objectives, prereq, evaluation, makeup] if x))
    return {
        "course_code": primary,
        "course_codes": codes,
        "title": title,
        "instructor": instructor,
        "description": desc,
        "objectives": objectives,
        "prerequisite_text": prereq,
        "evaluation_text": evaluation,
        "makeup_policy": makeup,
        "attendance": attendance,
        "has_midsem": has_midsem,
        "has_compre": has_compre,
        "projects": projects,
        "seminars": seminars,
        "quizzes": quizzes,
        "assignments": assignments,
        "projects_or_seminar": "yes" if projects == "yes" or seminars == "yes" else "unknown",
        "source": {"document": pdf_path.name, "pages": refs},
        "extraction_confidence": conf,
        "needs_verification": conf < 0.90,
        "search_text": search_text,
        "file_sha256": hashlib.sha256(pdf_path.read_bytes()).hexdigest(),
        "page_count": len(pages)
    }

In [ ]:
# CELL 6 — Parse every handout and write machine-readable data.
records = []
errors = []
for i, pdf in enumerate(pdf_paths, start=1):
    try:
        records.append(extract_handout(pdf))
    except Exception as e:
        errors.append({"file": pdf.name, "error": repr(e)})
    if i % 50 == 0 or i == len(pdf_paths):
        print(f"Parsed {i}/{len(pdf_paths)}")

(OUT / "handouts_raw.json").write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
(OUT / "handout_parse_errors.json").write_text(json.dumps(errors, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nTotal records:", len(records))
print("Parser errors:", len(errors))

In [ ]:
# CELL 7 — Validate extraction quality before the recommender sees the data.
missing_code = [r for r in records if not r.get("course_code")]
missing_title = [r for r in records if not r.get("title")]
weak = [r for r in records if r.get("needs_verification")]

print("PDFs:", len(pdf_paths))
print("Records:", len(records))
print("Parser errors:", len(errors))
print("Missing course code:", len(missing_code))
print("Missing title:", len(missing_title))
print("Needs verification flag:", len(weak))
print("\nMidsem:", Counter(r.get("has_midsem") for r in records))
print("Compre:", Counter(r.get("has_compre") for r in records))
print("Attendance:", Counter(r.get("attendance") for r in records))

assert len(errors) == 0, errors[:3]
assert len(records) == len(pdf_paths)
# Missing-title records are allowed only when the source PDF itself is blank/format-broken;
# the optional timetable merge below can fill many of these without inventing data.
print("\nParser integrity check: PASS")

# Show the weakest records for inspection, not automatic recommendation use.
for r in sorted(records, key=lambda x: x.get("extraction_confidence", 0))[:8]:
    print(r["source"]["document"], "|", r.get("course_code"), "|", r.get("title"), "| conf", r.get("extraction_confidence"))

In [ ]:
# CELL 8 — Optional timetable ingestion.
# This uses the supplied current-semester timetable only to fill title/units/offering metadata.
# It does NOT invent programme categories.

TIMETABLE = Path("/content/timetable.pdf") if Path("/content/timetable.pdf").exists() else None
if (TIMETABLE is None or not TIMETABLE.exists()) and Path("/mnt/data/source_docs/timetable.pdf").exists():
    TIMETABLE = Path("/mnt/data/source_docs/timetable.pdf")


def extract_timetable_text(pdf_path):
    with pymupdf.open(pdf_path) as doc:
        return [p.get_text("text") for p in doc]


def timetable_records(pdf_path, wanted_codes):
    pages = extract_timetable_text(pdf_path)
    found = defaultdict(list)
    code_pat = re.compile(r"^\s*([A-Z]{2,10}\s+[A-Z]\s?\d{3,4}[A-Z]?)\s*$", re.I)
    wanted = {norm_code(x) for x in wanted_codes if x}
    for page_no, text in enumerate(pages, 1):
        lines = [clean_space(x) for x in text.splitlines()]
        for i, line in enumerate(lines):
            m = code_pat.match(line)
            if not m: continue
            code = norm_code(m.group(1))
            if code not in wanted: continue
            # Next non-empty line is normally the course title.
            j = i + 1
            while j < len(lines) and not lines[j]: j += 1
            title = lines[j] if j < len(lines) else None
            # Find first few date/session-like entries; first is normally midsem, second compre.
            window = lines[i:min(len(lines), i+28)]
            dates = [x for x in window if re.search(r"\b\d{1,2}/\d{1,2}\s+[A-Z]{2,3}\d?\b", x)]
            # Units: find a compact credit row containing 1-5 integers/dashes.
            units = None
            for w in window[:10]:
                toks = w.split()
                nums = [t for t in toks if t.isdigit()]
                if len(toks) >= 1 and len(nums) >= 1 and len(nums) <= 5 and any(t == '-' for t in toks):
                    try:
                        units = int(nums[-1])
                        break
                    except Exception:
                        pass
            found[code].append({
                "course_code": code,
                "title": title,
                "units": units,
                "midsem": dates[0] if len(dates) >= 1 else None,
                "compre": dates[1] if len(dates) >= 2 else None,
                "source": {"document": pdf_path.name, "page": page_no},
                "raw_excerpt": " | ".join(window[:18])
            })
    return found

if TIMETABLE and TIMETABLE.exists():
    wanted_codes = [r.get("course_code") for r in records]
    tt = timetable_records(TIMETABLE, wanted_codes)
    (OUT / "timetable_raw.json").write_text(json.dumps(tt, indent=2, ensure_ascii=False), encoding="utf-8")
    filled = 0
    for r in records:
        hits = tt.get(r.get("course_code"), [])
        if not hits: continue
        best = next((h for h in hits if h.get("title")), hits[0])
        if not r.get("title") and best.get("title"):
            r["title"] = best["title"]; filled += 1
        r["timetable"] = hits
    print("Timetable loaded.")
    print("Codes found:", len(tt))
    print("Missing handout titles filled from timetable:", filled)
else:
    tt = {}
    print("Timetable not uploaded. This is allowed; timetable intelligence remains disabled.")

(OUT / "handouts_merged.json").write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
# CELL 9 — Build a clean recommendation catalog.
# One user-facing course card per primary handout code + title; source files are retained.

def build_catalog(recs):
    groups = {}
    for r in recs:
        code = r.get("course_code") or "UNKNOWN"
        title = clean_space(r.get("title") or "")
        key = (code, title.lower())
        if key not in groups:
            g = dict(r)
            g["source_files"] = [r["source"]["document"]]
            g["source_refs"] = [r["source"]]
            groups[key] = g
        else:
            g = groups[key]
            g["source_files"].append(r["source"]["document"])
            g["source_refs"].append(r["source"])
            # Prefer richer extracted text and stronger confidence.
            if len(r.get("search_text", "")) > len(g.get("search_text", "")):
                g["search_text"] = r["search_text"]
            for field in ["instructor", "description", "objectives", "prerequisite_text", "evaluation_text", "makeup_policy"]:
                if (not g.get(field)) and r.get(field):
                    g[field] = r[field]
            for field in ["attendance", "has_midsem", "has_compre", "projects", "seminars", "quizzes", "assignments", "projects_or_seminar"]:
                if g.get(field) == "unknown" and r.get(field) != "unknown":
                    g[field] = r[field]
            g["course_codes"] = unique(g.get("course_codes", []) + r.get("course_codes", []))
            g["needs_verification"] = bool(g.get("needs_verification") or r.get("needs_verification"))
            g["extraction_confidence"] = max(g.get("extraction_confidence", 0), r.get("extraction_confidence", 0))
            if "timetable" not in g and "timetable" in r:
                g["timetable"] = r["timetable"]
    out = []
    for g in groups.values():
        g["source_files"] = unique(g["source_files"])
        g["available_this_semester"] = True  # The supplied handout archive is the 2026-27 current-semester handout set.
        out.append(g)
    return sorted(out, key=lambda x: ((x.get("course_code") or ""), (x.get("title") or "")))

catalog = build_catalog(records)
(VERIFIED / "handouts.json").write_text(json.dumps(catalog, indent=2, ensure_ascii=False), encoding="utf-8")
print("Catalog cards:", len(catalog))
print("Unique primary codes:", len(set(r.get("course_code") for r in catalog)))
print("Cards requiring verification:", sum(r.get("needs_verification") for r in catalog))

## Academic-data boundary

The handouts alone cannot prove every programme rule. Therefore the notebook separates:

1. **source-backed course facts** extracted from the 540 handouts;
2. **registration-history filters** from the editable student profile;
3. **explicit timetable restrictions** when the timetable is loaded; and
4. **programme-role/category rules** only when a verified rules file is supplied.

This prevents the common failure mode of calling a course “DEL”, “HUEL” or “OPEL” merely because its topic looks relevant.

In [ ]:
# CELL 10 — Student profile.
# The defaults match the supplied Bulletin terminology for the user's dual-degree context.
# Completed/current courses are intentionally empty: enter only courses actually completed/current.

PROFILE_PATH = VERIFIED / "student_profile.json"

profile = {
    "campus": "Pilani",
    "admission_year": 2025,
    "programme": "M.Sc. Semiconductor and Nanoscience with B.E. Electrical & Electronics",
    "current_semester": 3,
    "completed_courses": [],
    "current_courses": [],
    "minor": None,
    "interests": ["AI/ML", "semiconductors", "electronics"],
}

if PROFILE_PATH.exists():
    try:
        saved = json.loads(PROFILE_PATH.read_text(encoding="utf-8"))
        profile.update(saved)
    except Exception:
        pass

print(json.dumps(profile, indent=2))

In [ ]:
# CELL 11 — Source-backed timetable restriction + profile exclusion helpers.
# The current 2026-27 timetable explicitly states that course numbers >=5000 are for 2026 admissions into FD/HD/PhD.
# We apply only that explicit restriction. Everything else stays source-dependent.


def norm_course_set(values):
    if values is None: return set()
    if isinstance(values, str):
        values = re.split(r"[,;\n]+", values)
    return {norm_code(v) for v in values if clean_space(str(v))}


def course_number(code):
    m = re.search(r"[A-Z](\d{3,4})", code or "")
    return int(m.group(1)) if m else None


def academic_gate(course, prof):
    reasons = []
    completed = norm_course_set(prof.get("completed_courses"))
    current = norm_course_set(prof.get("current_courses"))
    aliases = norm_course_set(course.get("course_codes", [])) | ({norm_code(course.get("course_code"))} if course.get("course_code") else set())

    if aliases & completed:
        return False, ["already in completed courses"]
    if aliases & current:
        return False, ["already in current courses"]

    n = course_number(course.get("course_code"))
    if n is not None and n >= 5000 and int(prof.get("admission_year", 0)) != 2026:
        return False, ["timetable restriction: course number >=5000 is for 2026 admissions"]

    return True, reasons


def evaluate_requirements(prof, rules=None):
    if not rules:
        return {"status": "not_loaded", "message": "No verified programme-rule file is loaded; programme category requirements are not guessed."}
    key = prof.get("programme_key") or prof.get("programme")
    rule = rules.get(key) if isinstance(rules, dict) else None
    if not rule:
        return {"status": "not_loaded", "message": "A matching verified programme rule was not found."}
    return {"status": "loaded", "rule": rule}

# Built-in source restriction test.
_test = {"course_code":"BITS F5001", "course_codes":["BITS F5001"]}
assert academic_gate(_test, {"admission_year":2025,"completed_courses":[],"current_courses":[]})[0] is False
assert academic_gate(_test, {"admission_year":2026,"completed_courses":[],"current_courses":[]})[0] is True
print("Academic gate tests: PASS")

In [ ]:
# CELL 12 — Lightweight TF-IDF-like semantic matcher with no external ML model.
# This makes the notebook stable and avoids model-download failures.

STOP = set("the a an and or for of to in on with from by is are be this that into as at course courses their your you i want need suggest find related no not have has based prefer preferably requirement requirements".split())
TOKEN_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9+\-]{2,}")


def tokens(text):
    vals = [x.lower() for x in TOKEN_RE.findall(text or "")]
    return [x for x in vals if x not in STOP]


def build_idf(docs):
    df = Counter()
    for d in docs:
        df.update(set(tokens(d)))
    n = max(1, len(docs))
    return {k: math.log((1+n)/(1+v)) + 1 for k,v in df.items()}

idf = build_idf([r.get("search_text", "") for r in catalog])


def cosine_query(query, text):
    q = Counter(tokens(query)); d = Counter(tokens(text))
    if not q or not d: return 0.0
    dot = nq = nd = 0.0
    for k, v in q.items():
        w = v * idf.get(k, 1.0); nq += w*w
        if k in d:
            dot += w * (d[k] * idf.get(k, 1.0))
    for k, v in d.items():
        w = v * idf.get(k, 1.0); nd += w*w
    return dot / math.sqrt(nq*nd) if nq and nd else 0.0


def parse_query(q):
    q0 = clean_space(q)
    u = q0.upper()
    category = next((x for x in ["CDC","DEL","HUEL","OPEL"] if re.search(rf"\b{x}\b", u)), None)
    wants = {
        "no_attendance": bool(re.search(r"\b(no|without|zero)\s+attendance|attendance\s+not\s+required", q0, re.I)),
        "no_midsem": bool(re.search(r"\b(no|without)\s+(a\s+)?mid[- ]?(sem|semester)", q0, re.I)),
        "no_compre": bool(re.search(r"\b(no|without)\s+(a\s+)?(?:compre|comprehensive)", q0, re.I)),
        "project": bool(re.search(r"\bproject[- ]?based|project\s+based|prefer\s+projects?", q0, re.I)),
        "seminar": bool(re.search(r"\bseminar", q0, re.I)),
        "quiz": bool(re.search(r"\bquiz", q0, re.I)),
        "assignment": bool(re.search(r"\bassignment", q0, re.I)),
        "makeup": bool(re.search(r"\bmake[- ]?up", q0, re.I)),
    }
    topic = re.sub(r"\b(CDC|DEL|HUEL|OPEL)\b", " ", q0, flags=re.I)
    topic = re.sub(r"\b(no|without|zero)\s+attendance|attendance\s+not\s+required", " ", topic, flags=re.I)
    topic = re.sub(r"\b(no|without)\s+(a\s+)?mid[- ]?(sem|semester)|\b(no|without)\s+(a\s+)?(?:compre|comprehensive)", " ", topic, flags=re.I)
    topic = re.sub(r"\bI\s+(want|need|prefer)\b|\bSuggest\b|\bFind\b|\brelated\s+to\b|\bwith\b|\band\b", " ", topic, flags=re.I)
    topic = clean_space(topic)
    return {"raw":q0,"category":category,"topic":topic,"preferences":wants}


def preference_pass(course, prefs):
    p = prefs["preferences"]
    if p["no_attendance"] and course.get("attendance") != "no":
        return False, "attendance is not explicitly verified as none"
    if p["no_midsem"] and course.get("has_midsem") != "no":
        return False, "a no-midsem claim is not explicitly verified"
    if p["no_compre"] and course.get("has_compre") != "no":
        return False, "a no-compre claim is not explicitly verified"
    if p["project"] and course.get("projects") != "yes":
        return False, "project-based evaluation is not explicitly verified"
    if p["seminar"] and course.get("seminars") != "yes":
        return False, "seminar component is not explicitly verified"
    if p["quiz"] and course.get("quizzes") != "yes":
        return False, "quiz component is not explicitly verified"
    if p["assignment"] and course.get("assignments") != "yes":
        return False, "assignment component is not explicitly verified"
    if p["makeup"] and not course.get("makeup_policy"):
        return False, "makeup policy is not explicitly present in the handout"
    return True, None


def recommend(query, prof, top_k=8, role_map=None):
    prefs = parse_query(query)
    results=[]; blocked=[]; category_unverified=False
    for course in catalog:
        ok, reasons = academic_gate(course, prof)
        if not ok:
            blocked.append((course,reasons[0])); continue
        if prefs["category"]:
            # A category is only accepted when an explicit verified role map is supplied.
            roles = set((role_map or {}).get(course.get("course_code"), []))
            if prefs["category"] not in roles:
                category_unverified = category_unverified or not role_map
                continue
        good, reason = preference_pass(course, prefs)
        if not good:
            blocked.append((course,reason)); continue
        score = cosine_query(prefs["topic"], course.get("search_text", ""))
        # Small deterministic boosts for explicit requested properties.
        if prefs["preferences"]["project"] and course.get("projects")=="yes": score += 0.15
        if prefs["preferences"]["seminar"] and course.get("seminars")=="yes": score += 0.10
        if prefs["preferences"]["quiz"] and course.get("quizzes")=="yes": score += 0.05
        if prefs["preferences"]["assignment"] and course.get("assignments")=="yes": score += 0.05
        if prefs["preferences"]["makeup"] and course.get("makeup_policy"): score += 0.05
        results.append((score,course))
    results.sort(key=lambda x:(-x[0], x[1].get("course_code") or ""))
    return prefs, results[:top_k], category_unverified

print("Matcher ready. Catalog vocabulary:", len(idf))

In [ ]:
# CELL 13 — Verify the real handout examples before the UI.
# These are source-grounded smoke tests on the actual 540-PDF archive.

by_code = {r.get("course_code"): r for r in catalog if r.get("course_code")}

checks = {}
for key in ["BITS F464", "BITS F471", "SNS F242", "CE G567", "SW E101"]:
    r = by_code.get(key)
    checks[key] = None if r is None else {
        "title": r.get("title"),
        "midsem": r.get("has_midsem"),
        "compre": r.get("has_compre"),
        "attendance": r.get("attendance"),
        "projects": r.get("projects"),
        "seminars": r.get("seminars"),
        "makeup_present": bool(r.get("makeup_policy"))
    }
print(json.dumps(checks, indent=2))

assert by_code.get("BITS F464", {}).get("title") == "Machine Learning"
assert by_code.get("BITS F471", {}).get("title") == "Introduction to Large Language Models"
assert any("SNS F242" in (r.get("course_codes") or []) and r.get("title") == "ELECTRONIC DEVICES" for r in catalog)
assert by_code.get("CE G567", {}).get("title") == "Highway Design"
assert by_code.get("SW E101", {}).get("title") == "Social Conduct 1"

# Real CHEM G551 handout check.
chem = by_code.get("CHEM G551")
if chem:
    print("\nCHEM G551:", {k:chem.get(k) for k in ["title","has_midsem","has_compre","quizzes","seminars","makeup_policy","attendance"]})
    assert chem.get("title") == "Advanced Organic Chemistry"
    assert chem.get("has_midsem") == "yes"
    assert chem.get("has_compre") == "yes"
    assert chem.get("quizzes") == "yes"
    assert chem.get("seminars") == "yes"
    assert chem.get("attendance") == "unknown"

# Test the profile exclusion gate.
q, res, _ = recommend("machine learning", {**profile, "completed_courses":["BITS F464"]})
assert all(r[1].get("course_code") != "BITS F464" for r in res)

# Topic matching smoke test on the real catalog.
q, res, _ = recommend("semiconductor", profile)
assert any("ELECTRONIC DEVICES" in (r[1].get("title") or "").upper() for r in res)

# Category query must refuse to guess when no verified role map is loaded.
q, res, category_unverified = recommend("Suggest OPEL related to AI", profile)
assert category_unverified is True
assert res == []

print("\nReal-data smoke tests: PASS")

In [ ]:
# CELL 14 — Clean notebook dashboard (ipywidgets, no Streamlit/tunnel).
# If widgets are unavailable in a non-Colab runtime, the notebook falls back to text mode instead of crashing.

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_OK = True
except Exception as e:
    WIDGETS_OK = False
    WIDGETS_ERROR = repr(e)

if WIDGETS_OK:
    campus_w = widgets.Dropdown(options=["Pilani","Goa","Hyderabad","Dubai"], value=profile.get("campus","Pilani"), description="Campus:")
    year_w = widgets.IntText(value=int(profile.get("admission_year",2025)), description="Admission:")
    programme_w = widgets.Text(value=profile.get("programme", ""), description="Programme:", layout=widgets.Layout(width="760px"))
    sem_w = widgets.IntSlider(value=int(profile.get("current_semester",3)), min=1, max=10, step=1, description="Semester:")
    completed_w = widgets.Textarea(value=", ".join(profile.get("completed_courses",[])), description="Completed:", layout=widgets.Layout(width="760px",height="70px"))
    current_w = widgets.Textarea(value=", ".join(profile.get("current_courses",[])), description="Current:", layout=widgets.Layout(width="760px",height="70px"))
    minor_w = widgets.Text(value=profile.get("minor") or "", description="Minor:")
    interests_w = widgets.Text(value=", ".join(profile.get("interests",[])), description="Interests:", layout=widgets.Layout(width="760px"))
    query_w = widgets.Textarea(value="Suggest courses related to semiconductors", description="Query:", layout=widgets.Layout(width="760px",height="70px"))
    save_btn = widgets.Button(description="Save Profile", button_style="primary")
    rec_btn = widgets.Button(description="Recommend", button_style="success")
    out = widgets.Output()

    def read_profile_widgets():
        return {
            "campus": campus_w.value,
            "admission_year": int(year_w.value),
            "programme": programme_w.value.strip(),
            "current_semester": int(sem_w.value),
            "completed_courses": [norm_code(x) for x in re.split(r"[,;\n]+", completed_w.value) if clean_space(x)],
            "current_courses": [norm_code(x) for x in re.split(r"[,;\n]+", current_w.value) if clean_space(x)],
            "minor": minor_w.value.strip() or None,
            "interests": [clean_space(x) for x in re.split(r"[,;\n]+", interests_w.value) if clean_space(x)],
        }

    def save_profile(_=None):
        global profile
        profile = read_profile_widgets()
        PROFILE_PATH.write_text(json.dumps(profile, indent=2, ensure_ascii=False), encoding="utf-8")
        with out:
            clear_output()
            print("Profile saved to:", PROFILE_PATH)
            print(json.dumps(profile, indent=2))

    def card(course, score):
        src = course.get("source", {})
        refs = src.get("pages", {}) if isinstance(src, dict) else {}
        ref_bits=[]
        if refs.get("course"): ref_bits.append(f"course details p.{refs['course']}")
        if refs.get("evaluation"): ref_bits.append(f"evaluation p.{refs['evaluation']}")
        if refs.get("makeup"): ref_bits.append(f"makeup p.{refs['makeup']}")
        ref = "; ".join(ref_bits) or "page not isolated"
        makeup = course.get("makeup_policy")
        verification = "<div style='margin-top:6px'><b>Verification flag:</b> needs verification</div>" if course.get("needs_verification") else ""
        makeup_html = f"<div style='margin-top:6px'><b>Makeup:</b> {html.escape(makeup[:500])}</div>" if makeup else ""
        return f"""
        <div style='border:1px solid #bbb;border-radius:12px;padding:14px;margin:10px 0;'>
          <div style='font-size:18px;font-weight:700'>{html.escape(course.get('course_code') or 'Unknown')} — {html.escape(course.get('title') or 'Title unavailable')}</div>
          <div style='margin-top:5px'>Match score: <b>{score:.3f}</b></div>
          <div style='margin-top:6px'>
            Midsem: <b>{course.get('has_midsem')}</b> &nbsp;|&nbsp;
            Compre: <b>{course.get('has_compre')}</b> &nbsp;|&nbsp;
            Attendance: <b>{course.get('attendance')}</b> &nbsp;|&nbsp;
            Project: <b>{course.get('projects')}</b> &nbsp;|&nbsp;
            Seminar: <b>{course.get('seminars')}</b>
          </div>
          <div style='margin-top:6px'><b>Instructor:</b> {html.escape(course.get('instructor') or 'Unknown')}</div>
          <div style='margin-top:6px'><b>Source:</b> {html.escape(src.get('document','unknown'))}; {html.escape(ref)}</div>
          {makeup_html}
          {verification}
        </div>
        """

    def recommend_click(_=None):
        global profile
        profile = read_profile_widgets()
        q, results, category_unverified = recommend(query_w.value, profile)
        with out:
            clear_output()
            print("Parsed query:", json.dumps(q, indent=2))
            if q.get("category") and category_unverified:
                display(HTML("<div style='padding:10px;border-radius:8px;background:#fff3cd;border:1px solid #e0c56e'><b>Category not verified:</b> no programme-role mapping is loaded, so the notebook will not call a course DEL/HUEL/OPEL.</div>"))
            if q.get("preferences",{}).get("no_attendance"):
                display(HTML("<div style='padding:10px;border-radius:8px;background:#f5f5f5;border:1px solid #bbb'><b>Attendance rule:</b> only explicitly verified 'no attendance' courses are allowed. Unknown is not treated as no.</div>"))
            if not results:
                print("No source-backed matches found under the current academic/profile gate.")
                if q.get("category"):
                    print("Add a verified programme-role map if you need DEL/HUEL/OPEL filtering.")
                return
            for score, course in results:
                display(HTML(card(course, score)))

    save_btn.on_click(save_profile)
    rec_btn.on_click(recommend_click)
    display(HTML("<h2>BITS Course Recommender</h2><p>Profile first. Then query in plain English. Academic/history exclusions are applied before topic matching.</p>"))
    display(widgets.VBox([
        campus_w, year_w, programme_w, sem_w, completed_w, current_w, minor_w, interests_w,
        widgets.HBox([save_btn, rec_btn]), query_w, out
    ]))
else:
    print("ipywidgets is unavailable in this runtime; the notebook remains runnable in text mode.")
    print("Use: recommend(QUERY, PROFILE) and inspect the returned course records.")
    print("Widget import error:", WIDGETS_ERROR)

In [ ]:
# CELL 15 — Final integrity report + export files.
report = {
    "handout_zip": str(HANDOUT_ZIP),
    "pdf_count": len(pdf_paths),
    "parsed_records": len(records),
    "parser_errors": len(errors),
    "catalog_cards": len(catalog),
    "missing_title": sum(not r.get("title") for r in records),
    "missing_code": sum(not r.get("course_code") for r in records),
    "verification_flags": sum(bool(r.get("needs_verification")) for r in catalog),
    "timetable_loaded": bool(TIMETABLE and TIMETABLE.exists()),
    "streamlit_used": False,
    "ngrok_used": False,
    "status": "PASS" if len(errors)==0 and len(records)==len(pdf_paths) else "FAIL"
}

(OUT / "final_integrity_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2))
assert report["status"] == "PASS"
print("\nFINAL INTEGRITY CHECK: PASS")
print("Generated files:")
for p in sorted(OUT.glob("*.json")) + sorted(VERIFIED.glob("*.json")):
    print(" -", p)

## Final usage

1. Upload your `handouts(1).zip` when Cell 3 asks for it.
2. Upload `bulletin.pdf`, `Academic-Regulations-2023.pdf`, and `timetable.pdf` when available; the notebook works without them, but it will not fabricate programme roles.
3. Run all cells top-to-bottom.
4. In the final dashboard, edit **Completed** and **Current** courses with your actual history.
5. Try queries such as:
   - `Suggest courses related to AI`
   - `Suggest courses related to semiconductors`
   - `Suggest project-based courses related to electronics`
   - `Suggest courses with no midsem`
   - `I want an OPEL related to AI`

For DEL/HUEL/OPEL, a verified programme-role mapping must exist before a course is allowed through that category filter. This is deliberate: the notebook refuses to invent academic categories.